In [ ]:
from utils_monoBP_single import *
from utils_nmodel import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.pyplot as plt
import math
import pandas as pd
import time
import sys

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
task_id = 0
data_obs = pd.read_csv(f"data_obs/data_obs_task{task_id}.csv")
data_obs = torch.tensor(data_obs.values, dtype = torch.float32).contiguous().to(device)

In [ ]:
x_obs = data_obs[:, 0]
y_obs = data_obs[:, 1]

In [ ]:
sigma = 0.1 
obs_size = 1000 

In [ ]:
psi = get_psi(x_obs, M)
A = get_A(M)

In [ ]:
design = psi @ torch.linalg.inv(A)
design = design.to(device)

In [ ]:
OLS = torch.linalg.solve(design.T @ design, design.T @ y_obs)

In [ ]:
samples_gibbs = torch.tensor(np.load(f'sample_res/theta_gibbs_task{task_id}.npy'), dtype=torch.float32)

In [ ]:
post_mean = samples_gibbs.mean(dim = 0)

In [ ]:
# Free GPU memory
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# model trained using prior proposal
model_pp = torch.load(f'prior_model/model_task{task_id}.pth', weights_only=False).to(device)

In [ ]:
model = ELU_LikeScoreMatchingNN(theta_dim = M + 1, x_dim = 2, obs_size = 1000, hidden_size= 64, num_layers = 3).to(device)
checkpoint = torch.load(f'nmodel_init/checkpoint_task{task_id}_trainsize{int(1e5)}.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
bias_lastlayer = checkpoint['bias_lastlayer']

with torch.no_grad(): 
    model.layers[-1].bias -= bias_lastlayer / obs_size

# Final result on posterior range, x and y axis same scale

## Zoom-in

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
num_grids = 30 
# Create a grid of x and y values
x = np.linspace(-0.96, -0.935, num_grids)
y = np.linspace(0.005, 0.037, 40) 
X, Y = np.meshgrid(x, y)

x_heat = np.linspace(-0.96, -0.935, 300)
y_heat = np.linspace(0.005, 0.037, 300) 
X_heat, Y_heat = np.meshgrid(x_heat, y_heat)

U_true = np.zeros(X.shape)
V_true = np.zeros(X.shape)
U_est = np.zeros(X.shape)
V_est = np.zeros(X.shape)
U_est_pp = np.zeros(X.shape)
V_est_pp = np.zeros(X.shape)

# calculate density
OLS = torch.linalg.solve(design.T @ design, design.T @ y_obs)
log_density = np.zeros(X_heat.shape)
for i in tqdm(range(X_heat.shape[0])):
    for j in range(X_heat.shape[1]):
        theta_tmp = post_mean.clone().to(device)
        theta_tmp[0] = X_heat[i, j]
        theta_tmp[1] = Y_heat[i, j]
        log_density[i, j] = ( -0.5 * (theta_tmp - OLS).view(1, -1) @ (design.T @ design)/sigma**2 @ (theta_tmp - OLS).view(-1, 1) ).item()
log_density -= log_density.max()
density = np.exp(log_density)

# calculate score
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        theta_tmp = post_mean.clone().to(device)
        theta_tmp[0] = X[i, j]
        theta_tmp[1] = Y[i, j]
        score_tmp = 1/sigma**2 * (design.T @ y_obs - design.T @ design @ theta_tmp).cpu()
        U_true[i, j] = score_tmp[0]
        V_true[i, j] = score_tmp[1]
        est_score_tmp = model(theta_tmp.view(1, -1), data_obs.view(1, -1)).detach().cpu().ravel()
        U_est[i, j] = est_score_tmp[0]
        V_est[i, j] = est_score_tmp[1]
        pp_est_score_tmp = model_pp(theta_tmp.view(1, -1), data_obs.view(1, -1)).detach().cpu().ravel()
        U_est_pp[i, j] = pp_est_score_tmp[0]
        V_est_pp[i, j] = pp_est_score_tmp[1]
        

# Compute the magnitude of the vector field for the background heatmap
magnitude_true = np.sqrt(U_true**2 + V_true**2)
magnitude_est = np.sqrt(U_est**2 + V_est**2)
magnitude_est_pp = np.sqrt(U_est_pp**2 + V_est_pp**2)

# Normalize vectors to have the same length
def normalize_vectors(U, V):
    magnitude = np.sqrt(U**2 + V**2)
    return U / magnitude, V / magnitude

U_true, V_true = normalize_vectors(U_true, V_true)
U_est, V_est = normalize_vectors(U_est, V_est)
U_est_pp, V_est_pp = normalize_vectors(U_est_pp, V_est_pp)

In [ ]:
plot_scale = 1500  

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

fig9, axes = plt.subplots(1, 3, figsize=(9, 4), constrained_layout=True)
ax0, ax1, ax2 = axes

# Density color normalization
norm_density = mcolors.Normalize(
    vmin=density.min(),
    vmax=density.max()
)

# Colormap for density
cmap_density = cm.Blues

arrow_color = "black"

# Subplot 0: Score estimated by SW proposal distribution
heatmap0 = ax0.pcolormesh(
    X_heat, Y_heat, density,
    shading="auto",
    cmap=cmap_density,
    norm=norm_density,
    rasterized=True
)
quiver0 = ax0.quiver(
    X, Y, U_est, V_est,
    angles="xy",
    scale=plot_scale,
    scale_units="xy",
    color=arrow_color,
    width=0.003
)
ax0.set_title("Estimated using Proposal")
ax0.set_xlabel(r"$\theta_0$")
ax0.set_ylabel(r"$\theta_1$")
ax0.grid(True, color="white", alpha=0.35, linewidth=0.8)

# Subplot 1: True score quiver plot
heatmap1 = ax1.pcolormesh(
    X_heat, Y_heat, density,
    shading="auto",
    cmap=cmap_density,
    norm=norm_density,
    rasterized=True
)
quiver1 = ax1.quiver(
    X, Y, U_true, V_true,
    angles="xy",
    scale=plot_scale,
    scale_units="xy",
    color=arrow_color,
    width=0.003
)
ax1.set_title("True score")
ax1.set_xlabel(r"$\theta_0$")
ax1.grid(True, color="white", alpha=0.35, linewidth=0.8)

# Subplot 2: Score estimated by prior proposal distribution
heatmap2 = ax2.pcolormesh(
    X_heat, Y_heat, density,
    shading="auto",
    cmap=cmap_density,
    norm=norm_density,
    rasterized=True
)
quiver2 = ax2.quiver(
    X, Y, U_est_pp, V_est_pp,
    angles="xy",
    scale=plot_scale,
    scale_units="xy",
    color=arrow_color,
    width=0.003
)
ax2.set_title("Estimated using Prior")
ax2.set_xlabel(r"$\theta_0$")
ax2.grid(True, color="white", alpha=0.35, linewidth=0.8)

# Keep equal aspect ratio
ax0.set_aspect("equal", adjustable="box")
ax1.set_aspect("equal", adjustable="box")
ax2.set_aspect("equal", adjustable="box")

# Hide repeated y tick labels
ax1.set_yticklabels([])
ax2.set_yticklabels([])

# Add one shared vertical colorbar on the right
cbar_density = fig9.colorbar(
    cm.ScalarMappable(norm=norm_density, cmap=cmap_density),
    ax=[ax0, ax1, ax2],
    orientation="vertical",
    fraction=0.035,
    pad=0.05,
    shrink=0.7
)
cbar_density.set_label("Density", rotation=270, labelpad=15)

plt.show()

# Band plot

In [ ]:
from torch.distributions import MultivariateNormal
import matplotlib.pyplot as plt

In [ ]:
def weighted_quantile(array, weight, quantile):
    # weight sums to 1
    sorter = np.argsort(array)
    sorted_array = array[sorter]
    sorted_weight = weight[sorter]
    index = np.searchsorted(np.cumsum(sorted_weight), quantile, side='left')
    return sorted_array[index]

def normalized_weight(log_w):
    tmp_w = (log_w - log_w.max()).exp()
    tmp_w = tmp_w / tmp_w.sum()

    thresh_id = 0
    while tmp_w.max() > 100 / log_w.shape[0]:
        thresh_id += 1
        clip = log_w.sort(descending=True).values[thresh_id]
        log_w = torch.clamp(log_w, max = clip)
        tmp_w = (log_w - log_w.max()).exp()
        tmp_w = tmp_w / tmp_w.sum()
    return tmp_w

In [ ]:
# ====== Sample results ====== #
pred_ys_BSL = torch.tensor( pd.read_csv(f'sample_res/pred_ys_BSL_task{task_id}.csv', header=None).values, 
                         dtype = torch.float32).contiguous()
pred_ys_ABCW1 = torch.tensor( pd.read_csv(f'sample_res/pred_ys_ABCW1_task{task_id}.csv', header=None).values, 
                         dtype = torch.float32).contiguous()

theta_r1_ABCW1 = torch.tensor(np.load(f"sample_res/theta_r1_ABCW1_task{task_id}.npy"), dtype=torch.float32)


# NLE res
theta_r1_NLE = torch.tensor(np.load(f"res_NLE/theta_post{task_id}.npy"), dtype=torch.float32)
A = get_A(M)
grids = torch.arange(0, 1.01, 0.01)
psi_grids = get_psi(grids, M)

if task_id == 6: # some chains failed
    theta_r1_NLE = theta_r1_NLE[250:]
if task_id == 9:
    theta_r1_NLE = theta_r1_NLE[:750]

pred_ys_NLE = psi_grids @ torch.linalg.inv(A) @ theta_r1_NLE.T

# NPE res
theta_r1_NPE = torch.tensor(np.load(f"res_NPE_newdeepsets/theta_post{task_id}.npy"), dtype=torch.float32)
pred_ys_NPE = psi_grids @ torch.linalg.inv(A) @ theta_r1_NPE.T

# ====== Get the weight for ABCW1 and NPE: w(\theta) \propto \frac{1}{q(\theta)}, where q(\theta) is the proposal density ====== #
theta_pre = torch.tensor(pd.read_csv(f"res_SW1_precond/theta_pre_task{task_id}.csv").values, dtype = torch.float32).contiguous()

lower = torch.zeros(M + 1) # .to(device)
lower[0] = a0
lower[1:] = a
upper = torch.zeros(M + 1) # .to(device)
upper[0] = b0
upper[1:] = b

actual_inf_rate = torch.ones(M + 1)
actual_inf_rate[-2:] = 2

inf_rate = torch.zeros(M + 1)
for i in range(M + 1):
    inf_rate[i] = get_inf_rate(mode = theta_pre.mean(dim = 0)[i].item(), std_orig = theta_pre.std(dim = 0)[i].item(),
                    lower = lower[i].item(), upper = upper[i].item(), actual_inf_rate = actual_inf_rate[i].item())


mean_theta = theta_pre.mean(dim = 0)
std_theta = inf_rate * theta_pre.std(dim = 0)

# the proposal distribution
dist = MultivariateNormal(loc = mean_theta, covariance_matrix = torch.diag(std_theta**2))

### Weight for NPE
log_weight_NPE = -dist.log_prob(theta_r1_NPE)
weight_NPE = normalized_weight(log_weight_NPE)

### Weight for ABCW1
log_weight_ABCW1 = -dist.log_prob(theta_r1_ABCW1)
weight_ABCW1 = normalized_weight(log_weight_ABCW1)

In [ ]:
pred_ys_gibbs = torch.tensor( pd.read_csv(f'sample_res/pred_ys_gibbs_task{task_id}.csv', header=None).values, 
                             dtype = torch.float32).contiguous()
pred_ys_fisherDR_cross = torch.tensor( pd.read_csv(f'sample_res_DebReg_fisher_cross/pred_ys_ann_task{task_id}.csv', header=None).values, 
                             dtype = torch.float32).contiguous()
pred_ys_nmodel_fisher = torch.tensor( pd.read_csv(f'sample_res_nmodel_fisher/pred_ys_ann_task{task_id}_trainsize100000.csv', header=None).values, 
                             dtype = torch.float32).contiguous()

pred_ys_nmodel_fisher_5x = torch.tensor( pd.read_csv(f'sample_res_nmodel_fisher/pred_ys_ann_task{task_id}_trainsize500000.csv', header=None).values, 
                             dtype = torch.float32).contiguous()

In [ ]:
### Plot error band
pred_list = [
    pred_ys_gibbs, pred_ys_fisherDR_cross, pred_ys_nmodel_fisher,
    pred_ys_nmodel_fisher_5x, pred_ys_ABCW1, pred_ys_BSL,
    pred_ys_NPE, pred_ys_NLE
]

weight_dict = {"ABC-W1": weight_ABCW1, "NPE": weight_NPE}
titles = ["True posterior", "single-model", "n-model", "n-model-5x",
          "ABC-W1", "BSL", "NPE", "NLE"]

fig, axes = plt.subplots(2, 4, figsize=(12, 7), sharex=True, sharey=True)
axes = axes.flatten()

grids = torch.arange(0, 1.01, 0.01)
true_f = torch.tanh(4.0 * grids - 2.0)

for i, (pred_ys, title) in enumerate(zip(pred_list, titles)):
    if title in ["ABC-W1", "NPE"]:
        weight = weight_dict[title]
        lower_bound = torch.zeros(pred_ys.shape[0])
        median = torch.zeros(pred_ys.shape[0])
        upper_bound = torch.zeros(pred_ys.shape[0])

        for j in range(pred_ys.shape[0]):
            lower_bound[j] = weighted_quantile(pred_ys[j], weight, 0.025)
            median[j] = weighted_quantile(pred_ys[j], weight, 0.5)
            upper_bound[j] = weighted_quantile(pred_ys[j], weight, 0.975)
    else:
        lower_bound = torch.quantile(pred_ys, 0.025, dim=1)
        median = torch.quantile(pred_ys, 0.5, dim=1)
        upper_bound = torch.quantile(pred_ys, 0.975, dim=1)

    ax = axes[i]

    # error relative to truth
    lower_err = lower_bound - true_f
    median_err = median - true_f
    upper_err = upper_bound - true_f

    ax.axhline(0, color="red", linewidth=0.8, linestyle="--")
    ax.fill_between(
        grids,
        lower_err,
        upper_err,
        alpha=0.5,
        label="95% posterior credible interval"
    )
    ax.plot(
        grids,
        median_err,
        color="black",
        linewidth=0.8,
        label="Posterior median"
    )

    ax.tick_params(axis="both", labelsize=12)
    ax.set_title(title, fontsize=16)
    ax.grid(True, alpha=0.7, linewidth=0.7)
    ax.set_xlabel(r"$x$", fontsize=14)

    if i % 4 == 0:
        ax.set_ylabel(r"$f(x; \theta) - \mathrm{E}[y \mid x]$", fontsize=14)

# Extract handles/labels from one of the plotted axes
handles, labels = axes[0].get_legend_handles_labels()

# Place the legend below the whole figure in one row
fig.legend(
    handles, labels,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=len(labels),
    fontsize=14,
    frameon=False
)

plt.tight_layout()
plt.show()